### import module

In [ ]:
import os
import numpy as np
import pandas as pd
import random

from typing import Dict

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss

from xgboost import XGBClassifier
import joblib
from pathlib import Path

### seed 고정

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

### 장고설정 및 모델 로딩

In [3]:
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true" # async 안전장치 해제 (장고 ORM은 sync 전용이라서 필요)
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "moathon.settings")

import django
django.setup()

from django.utils import timezone

from accounts.models import User, Moathon
from products.models import ProductOption 

### Product(부모모델) 자동 탐색
프로젝트마다 `ProductOption`의 부모 상품 모델명이 다를 수 있어,
`ProductOption.product` FK를 통해 **부모 상품 모델 찾음**

- `ProductModel` = `ProductOption.product.related_model`
- `product_id` = 부모 상품 PK

In [4]:
product_fk = ProductOption._meta.get_field("product")
ProductModel = product_fk.related_model

print("Detected Product model:", ProductModel)
print("ProductOption FK name:", product_fk.name)

Detected Product model: <class 'products.models.Product'>
ProductOption FK name: product


### 추천용 지표 함수 (Hit@K, Recall@K)

- **Hit@K**: 각 Moathon에서 Top-K 안에 정답 상품이 하나라도 있으면 1 하여 모든 moathon에 대해 평균낸 값

In [5]:
def hit_recall_at_k(pred_df: pd.DataFrame, k: int = 10) -> Dict[str, float]:
    '''
    pred_df columns: [moathon_id, y_true, score]
    (각 moathon_id에는 후보 product들이 여러 행)
    '''
    hits = []

    for mid, g in pred_df.groupby("moathon_id"):
        g = g.sort_values("score", ascending=False)
        topk = g.head(k)
        hit = 1.0 if topk["y_true"].sum() > 0 else 0.0
        hits.append(hit)
    return {"hit@k": float(np.mean(hits))}

### 옵션 기반 상품 집계 피처 생성

- 상품을 구분·비교할 수 있는 신호를 만들기 위해 **옵션 메타데이터(금리/기간 등)**로 상품 대표값을 만듬
- 상품 옵션들을 활용해서 요약 통계로 압축해서 사용

In [6]:
def build_product_agg_from_options() -> pd.DataFrame:
    # 존재하는 필드만 동적으로 선택
    all_fields = {f.name for f in ProductOption._meta.fields}

    base_fields = ["product_id"]
    cand_fields = ["intr_rate", "intr_rate2", "save_trm", "rsrv_type_nm", "intr_rate_type_nm"]
    fields = base_fields + [c for c in cand_fields if c in all_fields]

    qs = ProductOption.objects.values(*fields)
    opt = pd.DataFrame(list(qs))

    if opt.empty:
        # 반환 컬럼은 downstream에서 merge 시 편하도록 최소한 product_id만 보장
        return pd.DataFrame(columns=["product_id"])

    # save_trm은 CharField라 숫자로 변환해서 집계
    if "save_trm" in opt.columns:
        opt["save_trm_num"] = pd.to_numeric(opt["save_trm"], errors="coerce")

    agg_dict = {}

    # 금리 집계
    if "intr_rate2" in opt.columns:
        agg_dict["prod_max_intr_rate2"] = ("intr_rate2", "max")
        agg_dict["prod_mean_intr_rate2"] = ("intr_rate2", "mean")
    if "intr_rate" in opt.columns:
        agg_dict["prod_max_intr_rate"] = ("intr_rate", "max")
        agg_dict["prod_mean_intr_rate"] = ("intr_rate", "mean")

    # 기간/옵션 수 집계
    if "save_trm" in opt.columns:
        agg_dict["prod_min_save_trm"] = ("save_trm_num", "min")
        agg_dict["prod_max_save_trm"] = ("save_trm_num", "max")
        agg_dict["prod_cnt_options"] = ("save_trm_num", "count")

    agg = opt.groupby("product_id", as_index=False).agg(**agg_dict) if agg_dict else opt[["product_id"]].drop_duplicates()

    # 적립유형(정액/자유) 플래그
    if "rsrv_type_nm" in opt.columns:
        def _has_contains(series: pd.Series, keyword: str) -> int:
            s = series.fillna("").astype(str)
            return int(s.str.contains(keyword).any())

        rsrv_flags = opt.groupby("product_id")["rsrv_type_nm"].apply(
            lambda s: pd.Series({
                "has_rsrv_free": _has_contains(s, "자유"),   # "자유적립식" 
                "has_rsrv_fixed": _has_contains(s, "정액"),  # "정액적립식" 
            })
        ).reset_index()

        agg = agg.merge(rsrv_flags, on="product_id", how="left")
    else:
        agg["has_rsrv_free"] = 0
        agg["has_rsrv_fixed"] = 0

    # 금리유형(단리/복리 등) 플래그: intr_rate_type_nm 활용
    if "intr_rate_type_nm" in opt.columns:
        def _has_type(series: pd.Series, keyword: str) -> int:
            s = series.fillna("").astype(str)
            return int(s.str.contains(keyword).any())

        intr_type_flags = opt.groupby("product_id")["intr_rate_type_nm"].apply(
            lambda s: pd.Series({
                "has_simple_interest": _has_type(s, "단리"),
                "has_compound_interest": _has_type(s, "복리"),
            })
        ).reset_index()

        agg = agg.merge(intr_type_flags, on="product_id", how="left")
    else:
        agg["has_simple_interest"] = 0
        agg["has_compound_interest"] = 0

    # 혹시 merge로 NaN 생기면 0 처리)
    for c in ["has_rsrv_free", "has_rsrv_fixed", "has_simple_interest", "has_compound_interest"]:
        if c in agg.columns:
            agg[c] = agg[c].fillna(0).astype(int)

    return agg

### 학습 데이터셋 생성: (Moathon, Product) Pair + Negative Sampling

- Positive: `Moathon.product_option.product_id`
- Negative: 동일 Moathon에 대해 다른 상품 N개 랜덤 샘플
- neg_per_pos: 정답 상품이 아닌 데이터 생성 개수  

In [7]:
def build_stage1_pairs(
    neg_per_pos: int = 200,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    today = timezone.now().date()

    # Moathon + User + 선택된 product_id(정답) 로딩
    mo_qs = (Moathon.objects
             .select_related("user", "product_option", "product_option__product")
             .values(
                 "id",
                 "user_id",
                 "term_months",
                 "purpose",
                 "target_amount",
                 "start_amount",
                 "user__birth",
                 "user__gender",
                 "user__credit_score",
                 "user__assets",
                 "user__salary",
                 "user__average_monthly_spend",
                 "user__tender",
                 "product_option__product_id",
             ))
    base = pd.DataFrame(list(mo_qs))
    if base.empty:
        raise ValueError("Moathon 데이터가 비어 있습니다. 최소 1개 이상 필요합니다.")

    base = base.rename(columns={
        "id": "moathon_id",
        "product_option__product_id": "y_product_id",
        "user__gender": "gender",
        "user__credit_score": "credit_score",
        "user__assets": "assets",
        "user__salary": "salary",
        "user__average_monthly_spend": "average_monthly_spend",
        "user__tender": "tender",
    })

    # 유저 파생 피처
    birth = pd.to_datetime(base["user__birth"])
    base["age"] = ((pd.Timestamp(today) - birth).dt.days // 365).astype("int64")

    base["salary"] = base["salary"].fillna(0).astype("int64")
    base["average_monthly_spend"] = base["average_monthly_spend"].fillna(0).astype("int64")
    base["annual_spend"] = base["average_monthly_spend"] * 12
    base["disposable"] = base["salary"] - base["annual_spend"]
    base["spend_ratio"] = base["annual_spend"] / base["salary"].replace(0, 1)

    # 목표 파생
    base["need_amount"] = (base["target_amount"] - base["start_amount"]).clip(lower=0)
    base["need_per_month"] = base["need_amount"] / base["term_months"].replace(0, 1)

    # 상품 풀
    all_products = np.array(list(ProductModel.objects.values_list("id", flat=True)))
    if len(all_products) < 2:
        raise ValueError("상품(ProductModel) 개수가 너무 적습니다. 최소 2개 이상 필요합니다.")

    rows = []
    for _, r in base.iterrows():
        pos_pid = int(r["y_product_id"])
        # negative 후보
        neg_cand = all_products[all_products != pos_pid]
        n_neg = min(len(neg_cand), neg_per_pos)
        neg_pids = rng.choice(neg_cand, size=n_neg, replace=False)

        rows.append((int(r["moathon_id"]), int(r["user_id"]), pos_pid, 1))
        rows += [(int(r["moathon_id"]), int(r["user_id"]), int(pid), 0) for pid in neg_pids]

    pair = pd.DataFrame(rows, columns=["moathon_id", "user_id", "product_id", "y"])

    # base 피처 붙이기
    feat_cols = [
        "moathon_id", "user_id",
        "age", "gender", "credit_score", "assets",
        "salary", "average_monthly_spend", "tender",
        "annual_spend", "disposable", "spend_ratio",
        "term_months", "purpose", "target_amount", "start_amount",
        "need_amount", "need_per_month",
    ]
    df = pair.merge(base[feat_cols], on=["moathon_id", "user_id"], how="left")

    # 옵션 기반 상품 집계 피처 merge
    prod_agg = build_product_agg_from_options()
    df = df.merge(prod_agg, left_on="product_id", right_on="product_id", how="left")

    return df

### 데이터 생성 실행

In [8]:
df = build_stage1_pairs(neg_per_pos=20, seed=SEED)
print(df.shape)
df.head()

(3360000, 31)


,moathon_id,user_id,product_id,y,age,gender,credit_score,assets,salary,average_monthly_spend,...,prod_mean_intr_rate2,prod_max_intr_rate,prod_mean_intr_rate,prod_min_save_trm,prod_max_save_trm,prod_cnt_options,level_1_x,rsrv_type_nm,level_1_y,intr_rate_type_nm
0,1,1,73,1,30,0,821,7362260,44284078,2115579,...,2.87,3.2,2.87,6,36,8,has_rsrv_free,0,has_simple_interest,1
1,1,1,73,1,30,0,821,7362260,44284078,2115579,...,2.87,3.2,2.87,6,36,8,has_rsrv_free,0,has_compound_interest,1
2,1,1,73,1,30,0,821,7362260,44284078,2115579,...,2.87,3.2,2.87,6,36,8,has_rsrv_fixed,0,has_simple_interest,1
3,1,1,73,1,30,0,821,7362260,44284078,2115579,...,2.87,3.2,2.87,6,36,8,has_rsrv_fixed,0,has_compound_interest,1
4,1,1,312,0,30,0,821,7362260,44284078,2115579,...,1.80,2.8,1.80,1,36,12,has_rsrv_free,0,has_simple_interest,1


### 전처리/학습: XGBClassifier (binary) + Group split(user_id)

- split: 유저 단위 (`GroupShuffleSplit`)
- 목표: test에서 각 Moathon의 후보 상품들에 대해 점수를 매기고, 정답 상품이 Top-K에 들어가는지 측정

In [9]:
target = "y"
group_col = "user_id"

# 범주/수치 컬럼 정의 
cat_cols = ["gender", "tender", "purpose"]
num_cols = [
    "age", "credit_score", "assets", "salary", "average_monthly_spend",
    "annual_spend", "disposable", "spend_ratio",
    "term_months", "target_amount", "start_amount", "need_amount", "need_per_month",
    # 옵션 기반 상품 집계
    "prod_max_intr_rate2", "prod_mean_intr_rate2",
    "prod_max_intr_rate", "prod_mean_intr_rate",
    "prod_min_save_trm", "prod_max_save_trm", "prod_cnt_options",
    "has_rsrv_free", "has_rsrv_fixed",
]
cat_cols = [c for c in cat_cols if c in df.columns]
num_cols = [c for c in num_cols if c in df.columns]

X = df[cat_cols + num_cols]
y = df[target].astype(int)
groups = df[group_col].astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(splitter.split(X, y, groups=groups))

X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

meta_te = df.iloc[te_idx][["moathon_id", "product_id", "y"]].copy()

# 전처리: 범주=OneHot + 결측=UNK, 수치=median impute
preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
        ]), num_cols),
    ],
    remainder="drop",
)

# 불균형 보정
pos = int(y_tr.sum())
neg = int(len(y_tr) - pos)
scale_pos_weight = (neg / pos) if pos > 0 else 1.0
print("pos:", pos, "neg:", neg, "scale_pos_weight:", round(scale_pos_weight, 3))

model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=2,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    scale_pos_weight=scale_pos_weight,
)

pipe = Pipeline([("prep", preprocess), ("xgb", model)])
pipe.fit(X_tr, y_tr)

# 평가
proba_te = pipe.predict_proba(X_te)[:, 1]
print("Test logloss:", log_loss(y_te, proba_te))

pred_df = meta_te.rename(columns={"y": "y_true"}).copy()
pred_df["score"] = proba_te

for k in [1, 3, 5, 10]:
    m = hit_recall_at_k(pred_df[["moathon_id","y_true","score"]], k=k)
    print(f"K={k}  hit@k={m['hit@k']:.4f}")

pos: 127992 neg: 2559840 scale_pos_weight: 20.0
Test logloss: 0.17697369183334222
K=1  hit@k=0.7611
K=3  hit@k=0.7611
K=5  hit@k=0.8507
K=10  hit@k=0.9019


### 추천(추론): 특정 Moathon의 입력(유저+목표)으로 전 상품 점수화 → Top-K 상품 후보

- 실제 서비스에서는 사용자가 목표 입력하면 `Moathon`을 만들기 전이라도 동일 입력을 구성가능
- 여기서는 편의상 기존 Moathon 하나를 골라서 Top-K를 출력.

In [10]:
def build_single_query_row_from_moathon(moathon_id: int) -> pd.DataFrame:
    m = (Moathon.objects
         .select_related("user")
         .get(id=moathon_id))
    u = m.user
    today = timezone.now().date()
    age = (today - u.birth).days // 365

    salary = u.salary or 0
    avg_spend = u.average_monthly_spend or 0
    annual_spend = avg_spend * 12
    disposable = salary - annual_spend
    spend_ratio = annual_spend / (salary if salary != 0 else 1)

    need_amount = max(int(m.target_amount) - int(m.start_amount), 0)
    need_per_month = need_amount / (m.term_months if m.term_months else 1)

    row = {
        "age": age,
        "gender": u.gender,
        "credit_score": u.credit_score,
        "assets": u.assets,
        "salary": salary,
        "average_monthly_spend": avg_spend,
        "tender": u.tender,
        "annual_spend": annual_spend,
        "disposable": disposable,
        "spend_ratio": spend_ratio,
        "term_months": m.term_months,
        "purpose": m.purpose,
        "target_amount": int(m.target_amount),
        "start_amount": int(m.start_amount),
        "need_amount": need_amount,
        "need_per_month": need_per_month,
    }
    return pd.DataFrame([row])

def recommend_topk_products_for_row(
    pipe: Pipeline,
    user_goal_row: pd.DataFrame,
    top_k: int = 10,
) -> pd.DataFrame:
    # 후보 상품 DF (옵션 집계 포함)
    prod_ids = list(ProductModel.objects.values_list("id", flat=True))
    cand = pd.DataFrame({"product_id": prod_ids})

    prod_agg = build_product_agg_from_options()
    cand = cand.merge(prod_agg, on="product_id", how="left")

    # cross join
    user_goal_row = user_goal_row.copy()
    user_goal_row["key"] = 1
    cand["key"] = 1
    pair = user_goal_row.merge(cand, on="key").drop(columns=["key"])

    # pipe가 학습 때 사용한 컬럼이 없는 경우를 대비해, X 구성
    X_pair = pair[cat_cols + num_cols]  # cat_cols/num_cols는 학습 때 정의된 전역 값
    score = pipe.predict_proba(X_pair)[:, 1]

    out = pair[["product_id"]].copy()
    out["score"] = score
    return out.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)

# 테스트: 임의의 Moathon 1개 선택
sample_moathon_id = int(df["moathon_id"].iloc[0])
query_row = build_single_query_row_from_moathon(sample_moathon_id)

topk = recommend_topk_products_for_row(pipe, query_row, top_k=10)
topk

,product_id,score
0,73,0.985649
1,73,0.985649
2,73,0.985649
3,73,0.985649
4,74,0.985649
5,74,0.985649
6,74,0.985649
7,74,0.985649
8,138,0.966441
9,138,0.966441


### 상품명과 함께 반환

In [11]:
for cand in ["name", "product_name", "fin_prdt_nm", "title"]:
    if cand in [f.name for f in ProductModel._meta.fields]:
        name_field = cand
        break


mapping = dict(ProductModel.objects.values_list("id", name_field))
topk["product_name"] = topk["product_id"].map(mapping)
topk


,product_id,score,product_name
0,73,0.985649,e-정기예금
1,73,0.985649,e-정기예금
2,73,0.985649,e-정기예금
3,73,0.985649,e-정기예금
4,74,0.985649,스마트정기예금
5,74,0.985649,스마트정기예금
6,74,0.985649,스마트정기예금
7,74,0.985649,스마트정기예금
8,138,0.966441,e-정기예금
9,138,0.966441,e-정기예금


In [12]:
ART_DIR = Path("recommendations")
ART_DIR.mkdir(exist_ok=True)

# pipe = Pipeline([("prep", preprocess), ("xgb", model)])  # 이미 학습된 객체
joblib.dump(pipe, ART_DIR / "stage1_pipe.joblib")

# 학습에 사용한 컬럼 목록도 저장(추론 시 동일 컬럼 구성 체크용)
joblib.dump(
    {"cat_cols": cat_cols, "num_cols": num_cols},
    ART_DIR / "stage1_feature_spec.joblib"
)

print("Saved:", ART_DIR / "stage1_pipe.joblib")
print("Saved:", ART_DIR / "stage1_feature_spec.joblib")

Saved: recommendations\stage1_pipe.joblib
Saved: recommendations\stage1_feature_spec.joblib
